# Week 1 Day 3: Evaluation Harness + Stage 1 Fine-tuning Launch
## FIUBench Reproducibility Notebook

**Objective:** Build unified evaluation harness, configure and launch Stage 1 fine-tuning.

**Success Criteria:**
1. Repo cloned and environment ready
2. SFHQ images downloaded
3. Unified evaluation harness written (Rouge-L, Exact Match, KS-Test, MIA, APE)
4. Stage 1 fine-tuning configured and launched (lr=2e-5, seed=0)
5. Monitoring script running (Rouge-L + GPT-Eval on S_F at each checkpoint)
6. Results table templates created with placeholders


## Clone Repo and Setup Project root

In [ ]:
import os
from pathlib import Path

# Clone repo if not already present
if not os.path.exists('/content/FIUBench_Reproducing'):
    print("Cloning repo...")
    os.system('git clone https://github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing')
    print("✅ Clone complete")
else:
    print("✅ Repo already present — pulling latest...")
    os.system('git -C /content/FIUBench_Reproducing pull')

# Try Colab Drive mount (optional, for saving checkpoints)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    in_colab = True
except ImportError:
    in_colab = False

# Resolve PROJECT_ROOT
colab_root = '/content/FIUBench_Reproducing'
local_root = '/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/fiubench_reproducibility'
PROJECT_ROOT = colab_root if os.path.exists(colab_root) else local_root

assert os.path.exists(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print(f"✅ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✅ In Colab: {in_colab}")


Mounted at /content/drive
✅ PROJECT_ROOT: /content/FIUBench_Reproducing
✅ In Colab: True


## GPU Check

In [2]:
import torch

print("\n" + "="*60)
print("GPU STATUS")
print("="*60)

if torch.cuda.is_available():
    print(f"✅ CUDA available")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA version: {torch.version.cuda}")
    device = "cuda"
else:
    print("⚠️ No GPU — switch runtime to GPU before proceeding")
    device = "cpu"

print(f"✅ PyTorch: {torch.__version__}")
print("="*60 + "\n")



GPU STATUS
✅ CUDA available
   GPU: NVIDIA L4
   VRAM: 23.7 GB
   CUDA version: 12.8
✅ PyTorch: 2.10.0+cu128



## Install Dependencies

In [3]:
import subprocess
import sys
import torch

print("Installing dependencies...")

# Install everything EXCEPT transformers first
deps = [
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1",
    "flash-attn --no-build-isolation",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "python-dotenv",
    "openai",
    "hydra-core",
    "scipy",
    "deepspeed",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

# Pin transformers LAST — xtuner upgrades it to 5.x, so we overwrite after
subprocess.run(
    f"{sys.executable} -m pip install -q transformers==4.48.0",
    shell=True, check=True
)

# Verify via subprocess (no import needed — avoids loading wrong version)
result = subprocess.run(
    [sys.executable, "-c", "import transformers; print(transformers.__version__)"],
    capture_output=True, text=True
)
tf_ver = result.stdout.strip()
assert tf_ver == "4.48.0", f"transformers version mismatch: got {tf_ver}"

print("✅ Dependencies installed")
print(f"✅ torch={torch.__version__}")
print(f"✅ transformers={tf_ver}")


Installing dependencies...
✅ Dependencies installed
✅ torch=2.10.0+cu128
✅ transformers=4.48.0


## Load Model + Processor

In [4]:
import subprocess, sys
subprocess.run(f"{sys.executable} -m pip uninstall -y torchvision", shell=True)
subprocess.run(f"{sys.executable} -m pip install --no-cache-dir torchvision==0.19.1", shell=True)
print("✅ torchvision reinstalled")

✅ torchvision reinstalled


In [5]:
import subprocess, sys, os
from huggingface_hub import snapshot_download

# 1. Enable the ultra-fast Rust transfer library
subprocess.run(f"{sys.executable} -m pip install -q hf_transfer", shell=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_TOKEN"] = "REMOVED_TOKEN"

# 2. Download to LOCAL Colab disk (Bypasses the Google Drive bottleneck entirely)
MODEL_DIR = "/content/llava_phi_model" 

print("Downloading model to local NVMe at maximum speed...")
snapshot_download(
    repo_id="xtuner/llava-phi-3-mini-hf",
    local_dir=MODEL_DIR,
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
    token=os.environ["HF_TOKEN"],
)
print("✅ Done! You can now load the model.")

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.31G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Done! You can now load the model.


## Download SFHQ Images

In [6]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dir.mkdir(parents=True, exist_ok=True)

existing = list(sfhq_dir.glob("*.jpg"))
if len(existing) >= 400:
    print(f"✅ SFHQ images already present: {len(existing)}")
else:
    print("Downloading SFHQ.zip from Hugging Face...")
    zip_path = hf_hub_download(
        repo_id="gray311/FIUBench",
        filename="SFHQ.zip",
        repo_type="dataset",
        token=os.environ.get("HF_TOKEN"),
    )
    print(f"Downloaded to: {zip_path}")

    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting SFHQ.zip...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    print("Extraction complete")

    found = list(extract_dir.rglob("*.jpg"))
    print(f"Found {len(found)} jpg files in zip")
    for f in found[:3]:
        print(f"  {f.relative_to(extract_dir)}")

    copied = 0
    for src in found:
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            copied += 1

    total = len(list(sfhq_dir.glob("*.jpg")))
    print(f"✅ Copied {copied} new images — {total} total in {sfhq_dir}")

SFHQ.zip:   0%|          | 0.00/146M [00:00<?, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--gray311--FIUBench/snapshots/d9314f80f7214fbb41da238dbd45d61564b35e31/SFHQ.zip
Extracting SFHQ.zip...
Extraction complete
Found 1000 jpg files in zip
  SFHQ/SFHQ_pt1_00047930.jpg
  SFHQ/SFHQ_pt1_00021383.jpg
  SFHQ/SFHQ_pt1_00041161.jpg
✅ Copied 1000 new images — 1000 total in /content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ


## Unified Evaluation Harness

All metrics derived directly from `FIUBench/evaluate_util.py`:

- **Exact Match** — keyword-based partial credit (`eval_exact_match`): checks each keyword from `qa_list[i]['keywords']`
- **Min-K** — weighted combo across k=[0.1…0.5] with weights [0.3,0.3,0.2,0.1,0.1]; returns `sum(exp(mean(bottom-k%)) * w)`
- **Min-K++** — same but log-probs normalized by per-token (mean, std) before selection
- **Truth Ratio** — `exp(gt_loss/token - mean(perturb_loss/token))` from `eval_perturbation_ratio`
- **Rouge-L** — for Extension 1 text-only tasks (not in FIUBench core)
- **KS-Test** — distribution separation test on Min-K scores (forget vs retain)

In [ ]:
import re
import math
import json as _json
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from rouge_score import rouge_scorer
from scipy import stats
import torch.nn.functional as F

# ── Load local dataset (has keywords + perturbed_answer; HF dataset has both empty) ──
_full_data_path = Path(PROJECT_ROOT) / "FIUBench" / "dataset" / "full.json"
with open(_full_data_path) as _f:
    FULL_DATA = [_json.loads(line) for line in _f if line.strip()]

FULL_DATA_BY_ID = {row["unique_id"]: row for row in FULL_DATA}
print(f"✅ Loaded local full.json: {len(FULL_DATA)} rows")

SFHQ_DIR = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"

def resolve_image(image_path_str: str) -> Path:
    filename = Path(image_path_str).name
    return SFHQ_DIR / filename

# ── Rouge-L ───────────────────────────────────────────────────────────────────
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def rouge_l(prediction: str, reference: str) -> float:
    return _rouge.score(reference, prediction)["rougeL"].fmeasure

# ── Exact Match ───────────────────────────────────────────────────────────────
def exact_match(prediction: str, keywords: list) -> float:
    if not keywords:
        return 0.0
    score = sum(1.0 / len(keywords) for key in keywords if key.lower() in prediction.lower())
    return min(1.0, score)

# ── Min-K and Min-K++ ─────────────────────────────────────────────────────────
MINK_RATIOS  = [0.1, 0.2, 0.3, 0.4, 0.5]
MINK_WEIGHTS = [0.3, 0.3, 0.2, 0.1, 0.1]

@torch.no_grad()
def _get_answer_token_logprobs(question: str, answer: str, image_path: str):
    img_path = resolve_image(image_path)
    if not img_path.exists():
        return None, None, None
    img = Image.open(img_path).convert("RGB")
    prompt    = f"<image>\nUSER: {question}\nASSISTANT:"
    full_text = prompt + " " + answer
    inputs     = processor(text=full_text, images=img, return_tensors="pt").to(model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], pixel_values=inputs["pixel_values"])
    answer_logits = out.logits[0][prompt_len - 1: -1, :]
    answer_ids    = inputs["input_ids"][0][prompt_len:]
    log_probs = F.log_softmax(answer_logits.float(), dim=-1)
    probs     = log_probs.exp()
    token_lp  = log_probs.gather(-1, answer_ids.unsqueeze(-1)).squeeze(-1)
    mu        = (probs * log_probs).sum(-1)
    sigma     = ((probs * log_probs.pow(2)).sum(-1) - mu.pow(2)).clamp(min=1e-8).sqrt()
    result = (token_lp.cpu().numpy(), mu.cpu().numpy(), sigma.cpu().numpy())
    del out, answer_logits, log_probs, probs, token_lp, mu, sigma, inputs
    torch.cuda.empty_cache()
    return result

def mink(question: str, answer: str, image_path: str) -> float:
    token_lp, _, _ = _get_answer_token_logprobs(question, answer, image_path)
    if token_lp is None or len(token_lp) == 0:
        return float("nan")
    scores = [float(np.exp(np.mean(np.sort(token_lp)[:max(1, int(len(token_lp)*r))]))) for r in MINK_RATIOS]
    return sum(s * w for s, w in zip(scores, MINK_WEIGHTS))

def mink_plus_plus(question: str, answer: str, image_path: str) -> float:
    token_lp, mu, sigma = _get_answer_token_logprobs(question, answer, image_path)
    if token_lp is None or len(token_lp) == 0:
        return float("nan")
    normalized = (token_lp - mu) / (sigma + 1e-8)
    scores = [float(np.exp(np.mean(np.sort(normalized)[:max(1, int(len(normalized)*r))]))) for r in MINK_RATIOS]
    return sum(s * w for s, w in zip(scores, MINK_WEIGHTS))

# ── Truth Ratio ───────────────────────────────────────────────────────────────
@torch.no_grad()
def _answer_loss_per_token(question: str, answer: str, image_path: str) -> float:
    img_path = resolve_image(image_path)
    if not img_path.exists():
        return float("nan")
    img = Image.open(img_path).convert("RGB")
    prompt    = f"<image>\nUSER: {question}\nASSISTANT:"
    full_text = prompt + " " + answer
    inputs     = processor(text=full_text, images=img, return_tensors="pt").to(model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
    labels = inputs["input_ids"].clone()
    labels[0, :prompt_len] = -100
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], pixel_values=inputs["pixel_values"], labels=labels)
    loss = out.loss.item()
    del out, inputs, labels
    torch.cuda.empty_cache()
    return loss

def truth_ratio(question: str, answer: str, perturbed_answers: list, image_path: str) -> float:
    if not perturbed_answers:
        return float("nan")
    gt_loss = _answer_loss_per_token(question, answer, image_path)
    if math.isnan(gt_loss):
        return float("nan")
    perturb_losses = [pl for pl in [_answer_loss_per_token(question, pa, image_path) for pa in perturbed_answers] if not math.isnan(pl)]
    if not perturb_losses:
        return float("nan")
    return math.exp(gt_loss - float(np.mean(perturb_losses)))

# ── KS-Test ───────────────────────────────────────────────────────────────────
def ks_test(forget_scores: list, retain_scores: list) -> dict:
    f = [s for s in forget_scores if not math.isnan(s)]
    r = [s for s in retain_scores if not math.isnan(s)]
    if len(f) < 2 or len(r) < 2:
        return {"ks_stat": float("nan"), "p_value": float("nan")}
    res = stats.ks_2samp(f, r)
    return {"ks_stat": res.statistic, "p_value": res.pvalue}

# ── Smoke test ────────────────────────────────────────────────────────────────
print("Running smoke test (using local full.json)...")
row      = FULL_DATA[0]
q        = row["qa_list"][0]["question"]
a        = row["qa_list"][0]["answer"]
kws      = row["qa_list"][0]["keywords"]
perturbs = row["qa_list"][0].get("perturbed_answer", [])
img_p    = row["image_path"]

print(f"  Q: {q}")
print(f"  A: {a[:70]}...")
print(f"  keywords: {kws}")
print(f"  # perturbed answers: {len(perturbs)}")

em = exact_match(a, kws)
print(f"  EM (gold vs keywords): {em:.3f}  {'✅' if em > 0 or not kws else '⚠️ keywords not in answer'}")
print(f"  Rouge-L (gold vs gold): {rouge_l(a, a):.3f}")

# Min-K, Min-K++, Truth Ratio need model loaded — skip gracefully if not
_model_loaded = 'model' in dir() and 'processor' in dir() and 'tokenizer' in dir()
if _model_loaded:
    mk  = mink(q, a, img_p)
    mk2 = mink_plus_plus(q, a, img_p)
    print(f"  Min-K:   {mk:.3e}  (pretrained baseline; paper reports ~0.034 POST Stage-1 fine-tuning)")
    print(f"  Min-K++: {mk2:.3e}")
    tr = truth_ratio(q, a, perturbs, img_p)
    print(f"  Truth ratio: {tr:.4f}  {'(expect < 1 before fine-tuning)' if not math.isnan(tr) else '(skipped)'}")
else:
    print("  Min-K:   skipped (run cell b208b15a to load model first)")
    print("  Min-K++: skipped")
    print("  Truth ratio: skipped")

print("\n✅ Eval harness smoke test passed")


## Stage 1 - Fine-tuning

### Run FIUBench finetune.py — Paper methodology
**Hyperparameters (from paper Table 7):**
- lr=2e-5, seed=0, 10 epochs
- batch_size=8, gradient_accumulation_steps=16 (effective batch=128)
- LoRA r=0 (full fine-tuning of LM + projector, vision tower frozen)
- **max_length=512** (paper cutoff length for token truncation)
- data: full.json (400 identities × 20 QA = 8,000 pairs)

**⚠️ Hardware Compatibility Deviations:**
- Paper uses float16 precision on A100; Colab L4 uses bfloat16 (optimizer memory constraint)
- Paper uses flash_attention_2; Colab uses sdpa (GPU/driver compatibility)
- These are recorded for reproducibility notes

In [ ]:
import os
from pathlib import Path

FIUBENCH_DIR = Path(PROJECT_ROOT) / "FIUBench"
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["WANDB_MODE"] = "disabled"

# ── Patch finetune.py ─────────────────────────────────────────────────────────
# Use bfloat16 + sdpa: L4 has 22 GB — fp16 requires fp32 master weights in AdamW
# (~45 GB optimizer state alone), bfloat16 avoids this entirely.
# sdpa is the safe fallback; flash_attention_2 requires specific GPU/driver support.
ft_py = FIUBENCH_DIR / "finetune.py"
src = ft_py.read_text()
patched = src

# Ensure sdpa attention (safe on all GPUs with PyTorch >= 2.0)
patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')

# Ensure bfloat16 model dtype
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')

# Ensure bf16 mixed precision in Accelerate (no fp32 master weights)
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="fp16",\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)

if patched != src:
    ft_py.write_text(patched)
    print("✅ Patched finetune.py: sdpa + bfloat16 + bf16 mixed precision")
else:
    print("✅ finetune.py already at target settings")

content = ft_py.read_text()
assert 'attn_implementation="sdpa"' in content,       "FAILED: sdpa missing"
assert 'torch_dtype=torch.bfloat16' in content,       "FAILED: bfloat16 missing"
assert 'torch_dtype=torch.float16' not in content,    "FAILED: float16 still present"
assert 'mixed_precision="bf16"' in content,           "FAILED: mixed_precision=bf16 missing"
assert 'mixed_precision="fp16"' not in content,       "FAILED: fp16 still present"
print("✅ Verified: sdpa + bfloat16 + bf16 active in finetune.py")

# ── Patch modeling_llava.py ───────────────────────────────────────────────────
import re as _re
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _llava_src = _llava_path.read_text()
    _llava_patched = _re.sub(
        r"n_image_tokens != n_image_features",
        "n_image_tokens != image_features.shape[0]",
        _llava_src
    )
    # Cast selected_image_feature to projector dtype before linear layer
    # (CLIP loads in float32 by default; projector weights are bfloat16)
    _llava_patched = _llava_patched.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))"
    )
    if _llava_patched != _llava_src:
        _llava_path.write_text(_llava_patched)
        print("✅ Patched modeling_llava.py: fixed image token count check + dtype cast")
    else:
        print("✅ modeling_llava.py already patched")

# ── Patch finetune.py: fix end_training() called before final save ────────────
patched = ft_py.read_text()
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)
ft_py.write_text(patched)
print("✅ Patched finetune.py: end_training() moved after final save")

# ── Symlink SFHQ ──────────────────────────────────────────────────────────────
sfhq_src = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dst = FIUBENCH_DIR / "dataset" / "SFHQ"
if not sfhq_dst.exists():
    sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
    sfhq_dst.symlink_to(sfhq_src)
    print(f"✅ Symlinked {sfhq_dst} -> {sfhq_src}")
else:
    print(f"✅ SFHQ symlink/dir already exists at {sfhq_dst}")

# ── Verify model download exists ──────────────────────────────────────────────
MODEL_DIR = "/content/llava_phi_model"
assert Path(MODEL_DIR).exists(), f"Model not found at {MODEL_DIR} — run the download cell first"
model_gb = sum(f.stat().st_size for f in Path(MODEL_DIR).rglob("*") if f.is_file()) / 1e9
print(f"✅ Model at {MODEL_DIR}: {model_gb:.1f} GB")

# ── Disk usage check ──────────────────────────────────────────────────────────
import subprocess as _sp
_du = _sp.run("df -h /content", shell=True, capture_output=True, text=True)
print(_du.stdout.strip())

# ── Launch training ───────────────────────────────────────────────────────────
# Hyperparameters per paper Table 7: lr=2e-5, AdamW, batch=8, grad_accum=16 (effective=128),
# 10 epochs, LoRA r=0 (full fine-tuning of LM + projector; vision frozen).
# max_length=512: Paper cutoff length specification (input sequences truncated to 512 tokens)
LOCAL_CKPT = "/content/stage1_checkpoints"
DRIVE_CKPT = "/content/drive/MyDrive/fiubench_checkpoints/stage1"
Path(LOCAL_CKPT).mkdir(parents=True, exist_ok=True)

import subprocess, sys, time as _time

_cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29503 finetune.py "
    f"--config-name finetune_llava_phi "
    f"model_id={MODEL_DIR} "
    f"save_dir={LOCAL_CKPT} "
    f"save_steps=2310 "
    f"seed=0 lr=2e-5 "
    f"batch_size=8 "
    f"gradient_accumulation_steps=16 "
    f"max_length=512 "
    f"2>&1 | tee /tmp/ft_log.txt"
)

_t0 = _time.time()
_proc = subprocess.Popen(_cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
for _line in _proc.stdout:
    print(_line, end="", flush=True)
_proc.wait()
ret = _proc.returncode

_elapsed = _time.time() - _t0
_h, _m, _s = int(_elapsed // 3600), int((_elapsed % 3600) // 60), int(_elapsed % 60)
print(f"Exit code: {ret}")
print(f"Training time: {_h}h {_m}m {_s}s")

if ret == 0:
    Path(DRIVE_CKPT).mkdir(parents=True, exist_ok=True)
    print("Copying checkpoint to Drive...")
    os.system(f"rsync -ah --progress {LOCAL_CKPT}/ {DRIVE_CKPT}/")
    os.system(f"cp /tmp/ft_log.txt {DRIVE_CKPT}/training_log.txt")
    print(f"✅ Checkpoint backed up to {DRIVE_CKPT}")
    print(f"✅ Training log saved to {DRIVE_CKPT}/training_log.txt")
else:
    print(f"❌ Training failed (exit {ret}). Check /tmp/ft_log.txt")

## Rouge-L Evaluation of Fine-tuned Phi model on FIUBench Dataset --> Evaluated on retain5 subset

In [ ]:
import subprocess, json
from pathlib import Path

# ── Stage 1 Verification Eval ─────────────────────────────────────────────────
# Runs evaluate_util.py on Stage 1 checkpoint (no LoRA) on retain5.
# GPT-Eval skipped here — checked on Day 5 with OpenAI key.

FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
LOCAL_CKPT   = '/content/stage1_checkpoints'
VERIFY_DIR   = f'{LOCAL_CKPT}/stage1_verify'

print("=" * 60)
print("Stage 1 Verification — ROUGE-L on retain5")
print("Paper target: 93.3  |  Acceptable threshold: >= 88.0")
print("=" * 60)

proc = subprocess.Popen(
    ['python', 'evaluate_util.py', '--config-name', 'eval',
     f'model_path={LOCAL_CKPT}',
     'LoRA.r=0',
     f'save_dir={VERIFY_DIR}',
     'split_list=[retain5]',
     'eval_task=[eval_retain_log]',
     'robust_eval=[[rouge]]',
     'batch_size=1', 'perturb_batch_size=1',
     'overwrite=true',
     'hydra.run.dir=/tmp/hydra_stage1_verify'],
    cwd=FIUBENCH_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

# ── Parse and report ──────────────────────────────────────────────────────────
result_path = Path(VERIFY_DIR) / 'retain5_eval_retain_log.json'
if result_path.exists():
    with open(result_path) as f:
        data = json.load(f)
    rouge = data.get('rougeL_recall', float('nan')) * 100
    print(f"\n{'='*60}")
    print(f"  Stage 1 ROUGE-L (retain5): {rouge:.1f}")
    print(f"  Paper target:              93.3")
    if rouge >= 88.0:
        print(f"  ✅ Stage 1 verified — proceed to Stage 2")
    elif rouge >= 80.0:
        print(f"  ⚠️  ROUGE-L below 88 — weaker than paper but may still proceed")
        print(f"     Check: {DRIVE_CKPT}/training_log.txt")
    else:
        print(f"  ❌ ROUGE-L below 80 — Stage 1 likely failed. Do NOT proceed to Stage 2.")
    print('='*60)
else:
    print("❌ Result file not found — check eval output above for errors")


In [ ]:
import json
from pathlib import Path

result_path = Path('/content/stage1_checkpoints/stage1_verify/retain5_eval_retain_log.json')
with open(result_path) as f:
    data = json.load(f)

scores = list(data['rougeL_recall'].values())
scores_pct = [v * 100 for v in scores]

print(f"N samples: {len(scores_pct)}")
print(f"Mean ROUGE-L: {sum(scores_pct)/len(scores_pct):.1f}")
print(f"Min:  {min(scores_pct):.1f}")
print(f"Max:  {max(scores_pct):.1f}")

# Distribution
buckets = [0]*5
for s in scores_pct:
    if s < 20: buckets[0] += 1
    elif s < 40: buckets[1] += 1
    elif s < 60: buckets[2] += 1
    elif s < 80: buckets[3] += 1
    else: buckets[4] += 1
print(f"<20: {buckets[0]}, 20-40: {buckets[1]}, 40-60: {buckets[2]}, 60-80: {buckets[3]}, 80+: {buckets[4]}")


In [ ]:
rouge = data['rougeL_recall']
if isinstance(rouge, dict):
    vals = list(rouge.values())
    score = sum(vals) / len(vals) * 100
else:
    score = rouge * 100
print(f"Stage 1 ROUGE-L: {score:.1f}  (target: 93.3, threshold: 88.0)")
